# RelBench quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/snap-stanford/relbench/blob/relbench-hf/tutorials/quickstart.ipynb)

Load a RelBench dataset and task straight from the Hugging Face Hub, explore the
relational schema, and train a trivial baseline — all with **no per-dataset code**.

Run every cell top to bottom. In Colab the first cell installs RelBench; locally you can
skip it if RelBench is already installed.

In [ ]:
# Install RelBench (skip if already installed locally)
!pip install relbench

In [ ]:
import numpy as np
import pandas as pd

import relbench

## Load the dataset

In [ ]:
dataset = relbench.load_dataset("rel-f1")
db = dataset.get_db()

The dataset is a set of tables linked by a foreign-key graph — all described by its `manifest.yaml`:

In [ ]:
pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(table),
            "pkey": table.pkey_col,
            "time_col": table.time_col,
            "fkeys": ", ".join(table.fkey_col_to_pkey_table) or "—",
        }
        for name, table in db.table_dict.items()
    ]
)

In [ ]:
db.table_dict["results"].df.head()

## Tasks

List the tasks available for this dataset:

In [ ]:
relbench.get_task_names("rel-f1")

In [ ]:
task = relbench.load_task("rel-f1", "driver-position")
train_df = task.get_table("train").df
train_df.head()

## A trivial baseline

Predict the training-set mean for every test entity, and evaluate with the task's own metrics:

In [ ]:
test_table = task.get_table("test", mask_input_cols=False)
pred = np.full(len(test_table.df), train_df[task.target_col].mean())
metrics = task.evaluate(pred, test_table)
metrics